In [ ]:
import numpy as np
import ipywidgets as widgets
from IPython.display import display, HTML
from google.colab import output
import base64
import os
import random

# --- CARGAR LAS TRES IMÁGENES DE FORMA SEGURA ---
img1_base64 = ""
img2_base64 = ""
img3_base64 = ""

# 1. Imagen de Inicio
for filename in ["Wallpaper del momo tito calderon.jpg", "Wallpaper del momo tito calderon.JPG", "wallpaper.jpg"]:
    if os.path.exists(filename):
        with open(filename, "rb") as f:
            img1_base64 = base64.b64encode(f.read()).decode('utf-8')
        break

# 2. Imagen de Victoria
for filename in ["bradd pittin.jpg", "bradd pittin.JPG", "bradpitt.jpg"]:
    if os.path.exists(filename):
        with open(filename, "rb") as f:
            img2_base64 = base64.b64encode(f.read()).decode('utf-8')
        break

# 3. Imagen de Derrota (Tiempo agotado)
for filename in ["Ahhhhhhhhhhh.jpg", "Ahhhhhhhhhhh.JPG", "ahhhhh.jpg"]:
    if os.path.exists(filename):
        with open(filename, "rb") as f:
            img3_base64 = base64.b64encode(f.read()).decode('utf-8')
        break

# --- BANCO DE MAPAS EQUILIBRADOS Y 100% ACCESIBLES ---
banco_mapas = {
    1: [
        np.array([[0,2,1,2,0,2],[2,0,2,0,2,0],[1,2,0,2,1,2],[2,0,2,0,2,0],[0,2,1,2,0,2],[2,0,2,0,2,9]], dtype=int),
        np.array([[0,2,2,1,2,2],[2,0,2,2,0,2],[2,2,0,1,2,0],[1,0,1,0,2,2],[2,2,2,2,0,2],[0,2,0,2,2,9]], dtype=int)
    ],
    2: [
        np.array([[0,2,1,2,0,2,1],[2,0,2,0,2,0,2],[1,2,0,2,1,2,0],[2,0,2,0,2,0,2],[0,2,1,2,0,2,1],[2,0,2,0,2,0,2],[1,2,0,2,1,2,9]], dtype=int),
        np.array([[0,2,2,1,2,0,2],[2,0,2,2,0,2,1],[2,2,0,1,2,2,0],[1,0,1,0,2,0,2],[2,2,2,2,0,2,2],[0,2,0,2,2,1,0],[2,2,2,0,2,2,9]], dtype=int)
    ],
    3: [
        np.array([[0,2,1,2,0,2,1,2],[2,0,2,0,2,0,2,0],[1,2,0,2,1,2,0,2],[2,0,2,0,2,0,2,0],[0,2,1,2,0,2,1,2],[2,0,2,0,2,0,2,0],[1,2,0,2,1,2,0,2],[2,0,2,0,2,0,2,9]], dtype=int),
        np.array([[0,2,2,1,2,0,2,2],[2,0,2,2,0,2,1,0],[2,2,0,1,2,2,0,2],[1,0,1,0,2,0,2,1],[2,2,2,2,0,2,2,0],[0,2,0,2,2,1,0,2],[2,2,2,0,2,2,1,0],[0,2,2,2,0,2,2,9]], dtype=int)
    ]
}

TIEMPOS_NIVEL = {1: 50, 2: 40, 3: 30}

# --- ESTADO GLOBAL DEL JUEGO ---
nivel_actual = 1
puntuacion = 0
high_score = 0
pos_jugador = [0, 0]
tiempo_restante = 0
tablero_activo = None
tam_actual = 6
juego_iniciado = False
juego_activo = False

pantalla_principal = widgets.HTML()
contenedor_principal = widgets.VBox(layout=widgets.Layout(align_items='center'))

def contar_monedas_restantes():
    return np.sum(tablero_activo == 2)

def cargar_nivel():
    global tablero_activo, tam_actual, pos_jugador, tiempo_restante, juego_activo
    mapa_elegido = random.choice(banco_mapas[nivel_actual])
    tablero_activo = mapa_elegido.copy()
    tam_actual = tablero_activo.shape[0]
    tiempo_restante = TIEMPOS_NIVEL[nivel_actual]
    pos_jugador = [0, 0]
    juego_activo = True

def renderizar(mensaje_estado=""):
    global high_score, juego_activo, juego_iniciado

    if not juego_iniciado:
        img_tag = f"<div style='margin-bottom: 15px;'><img src='data:image/jpeg;base64,{img1_base64}' style='width: 100%; height: auto; border-radius: 10px; box-shadow: 0 4px 12px rgba(0,0,0,0.5); display: block;'></div>" if img1_base64 else ""

        pantalla_principal.value = (
            "<div style='font-family: \"Segoe UI\", Roboto, Helvetica, sans-serif; background: linear-gradient(135deg, #090d16 0%, #1e1b4b 100%); color: #f8fafc; padding: 30px; border-radius: 16px; max-width: 480px; text-align: center; box-shadow: 0 20px 40px rgba(0,0,0,0.6); border: 1px solid #38bdf8; box-sizing: border-box;'>"
            + img_tag +
            "<h1 style='color: #38bdf8; margin: 0 0 20px 0; font-size: 26px; letter-spacing: 1px; text-shadow: 0 2px 10px rgba(56,189,248,0.3);'>⚡ LABERINTO MATRICIAL</h1>"
            "<div style='background: rgba(30, 41, 59, 0.75); backdrop-filter: blur(10px); padding: 16px; border-radius: 12px; margin-bottom: 20px; text-align: left; font-size: 13px; border: 1px solid rgba(255,255,255,0.08); box-shadow: inset 0 1px 0 rgba(255,255,255,0.1);'>"
            "<b style='color: #f1f5f9; display: block; margin-bottom: 8px;'>🎮 Reglas del Juego:</b>"
            "<div style='color: #cbd5e1; line-height: 1.5;'>"
            "• Recoge todas las monedas (🪙) del tablero.<br>"
            "• La puerta (🚪) se habilitará al limpiar el mapa.<br>"
            "• El tiempo se reduce de forma estricta en cada nivel.<br>"
            "• Usa las <b>flechas del teclado</b> o <b>W, A, S, D</b>."
            "</div></div>"
            "<button onclick=\"google.colab.kernel.invokeFunction('notebook.iniciar_juego', [], {})\" "
            "style='background: linear-gradient(135deg, #3b82f6 0%, #1d4ed8 100%); color: white; border: none; padding: 12px 28px; font-size: 15px; font-weight: bold; border-radius: 30px; cursor: pointer; box-shadow: 0 4px 15px rgba(59, 130, 246, 0.4);'>"
            "🚀 Iniciar Partida"
            "</button></div>"
        )
        return

    if puntuacion > high_score:
        high_score = puntuacion

    color_tiempo = "#f87171" if tiempo_restante <= 10 else "#4ade80"
    html_content = (
        f"<div style='font-family: \"Segoe UI\", Roboto, Helvetica, sans-serif; background: linear-gradient(135deg, #0f172a 0%, #090d16 100%); color: #f8fafc; padding: 24px; border-radius: 16px; max-width: 500px; box-shadow: 0 20px 40px rgba(0,0,0,0.5); border: 1px solid #1e293b; box-sizing: border-box;'>"
        f"<div style='display: flex; justify-content: space-between; align-items: center; border-bottom: 1px solid #334155; padding-bottom: 12px; margin-bottom: 16px;'>"
        f"<h2 style='margin: 0; color: #38bdf8; font-size: 18px; letter-spacing: 0.5px;'>⚡ LABERINTO MATRICIAL</h2>"
        f"<span style='background: rgba(56, 189, 248, 0.1); color: #38bdf8; padding: 5px 12px; border-radius: 20px; font-size: 12px; font-weight: 600; border: 1px solid rgba(56, 189, 248, 0.2);'>Nivel {nivel_actual} / 3</span>"
        f"</div>"
        f"<div style='display: flex; justify-content: space-between; font-size: 13px; margin-bottom: 16px; background: rgba(30, 41, 59, 0.6); padding: 12px 16px; border-radius: 10px; border: 1px solid rgba(255,255,255,0.05);'>"
        f"<div>🪙 Puntos: <b style='color: #f1f5f9;'>{puntuacion}</b></div>"
        f"<div>🏆 Récord: <b style='color: #f1f5f9;'>{high_score}</b></div>"
        f"<div style='color: {color_tiempo};'>⏱️ Tiempo: <b>{tiempo_restante}s</b></div>"
        f"</div>"
    )

    if mensaje_estado:
        html_content += f"<div style='background: rgba(51, 65, 85, 0.8); color: #facc15; padding: 10px; border-radius: 8px; text-align: center; margin-bottom: 16px; font-weight: 600; font-size: 13px; border: 1px solid rgba(250,204,21,0.2);'>{mensaje_estado}</div>"

    html_content += f"<div style='grid-template-columns: repeat({tam_actual}, 40px); gap: 4px; justify-content: center; background: #020617; padding: 16px; border-radius: 12px; border: 1px solid #1e293b; box-shadow: inset 0 2px 6px rgba(0,0,0,0.5); display: grid;'>"

    for f in range(tam_actual):
        for c in range(tam_actual):
            celda_estilo = "width: 40px; height: 40px; display: flex; align-items: center; justify-content: center; font-size: 18px; border-radius: 6px; background: #1e293b;"
            contenido = ""

            if [f, c] == pos_jugador:
                contenido = "🤖"
                celda_estilo += " background: #0284c7; box-shadow: 0 0 10px rgba(2, 132, 199, 0.6);"
            elif tablero_activo[f, c] == 1:
                contenido = "🧱"
                celda_estilo += " background: #334155; opacity: 0.9;"
            elif tablero_activo[f, c] == 2:
                contenido = "🪙"
            elif tablero_activo[f, c] == 9:
                contenido = "🚪"
                if contar_monedas_restantes() == 0:
                    celda_estilo += " background: #16a34a; box-shadow: 0 0 12px #22c55e;"
                else:
                    celda_estilo += " background: #7f1d1d; opacity: 0.8;"
            else:
                contenido = ""

            html_content += f"<div style='{celda_estilo}'>{contenido}</div>"

    monedas_faltantes = contar_monedas_restantes()
    html_content += (
        f"</div>"
        f"<div style='margin-top: 16px; font-size: 12px; color: #94a3b8; text-align: center;'>"
        f"Monedas pendientes: <b style='color: #f8fafc;'>{monedas_faltantes}</b><br>"
        f"<i style='color: #64748b;'>Usa las flechas del teclado o W,A,S,D para moverte con fluidez.</i>"
        f"</div></div>"
    )
    pantalla_principal.value = html_content

def iniciar_juego(b=None):
    global juego_iniciado, puntuacion, nivel_actual
    juego_iniciado = True
    nivel_actual = 1
    puntuacion = 0
    cargar_nivel()
    renderizar()

def mover_teclado(direccion):
    global puntuacion, nivel_actual, juego_activo, juego_iniciado
    if not juego_iniciado or not juego_activo:
        return

    f_cambio, c_cambio = 0, 0
    if direccion == 'ArrowUp': f_cambio = -1
    elif direccion == 'ArrowDown': f_cambio = 1
    elif direccion == 'ArrowLeft': c_cambio = -1
    elif direccion == 'ArrowRight': c_cambio = 1

    nueva_f = pos_jugador[0] + f_cambio
    nueva_c = pos_jugador[1] + c_cambio

    if 0 <= nueva_f < tam_actual and 0 <= nueva_c < tam_actual:
        if tablero_activo[nueva_f, nueva_c] != 1:

            if tablero_activo[nueva_f, nueva_c] == 9:
                if contar_monedas_restantes() == 0:
                    if nivel_actual < 3:
                        nivel_actual += 1
                        puntuacion += 100
                        cargar_nivel()
                        renderizar(f"¡Excelente! Avanzando al Nivel {nivel_actual}...")
                        return
                    else:
                        juego_activo = False
                        img_win = f"<div style='margin-bottom: 15px;'><img src='data:image/jpeg;base64,{img2_base64}' style='width: 100%; height: auto; border-radius: 10px; box-shadow: 0 4px 12px rgba(0,0,0,0.5); display: block;'></div>" if img2_base64 else ""

                        pantalla_principal.value = (
                            "<div style='font-family: \"Segoe UI\", Roboto, Helvetica, sans-serif; background: linear-gradient(135deg, #090d16 100%, #064e3b 0%); color: #f8fafc; padding: 30px; border-radius: 16px; text-align: center; border: 2px solid #22c55e; max-width: 480px; box-shadow: 0 20px 40px rgba(0,0,0,0.6); box-sizing: border-box;'>"
                            + img_win +
                            "<h1 style='color: #4ade80; font-size: 26px; margin-bottom: 8px; text-shadow: 0 2px 10px rgba(74,222,128,0.4);'>🏆 ¡HAS GANADO EL JUEGO! 🏆</h1>"
                            "<p style='font-size: 14px; color: #cbd5e1; margin-bottom: 16px;'>¡Has dominado las matrices y completado todos los niveles con éxito!</p>"
                            "<div style='background: rgba(15, 23, 42, 0.85); padding: 14px; border-radius: 10px; margin: 15px 0; border: 1px solid rgba(255,255,255,0.1);'>"
                            f"<h3 style='margin: 0 0 5px 0; color: #38bdf8;'>Puntuación Final: {puntuacion} pts</h3>"
                            f"<p style='margin: 0; color: #94a3b8; font-size: 13px;'>Récord registrado: <b>{high_score}</b></p>"
                            "</div>"
                            "<button onclick=\"google.colab.kernel.invokeFunction('notebook.iniciar_juego', [], {})\" "
                            "style='background: linear-gradient(135deg, #22c55e 0%, #15803d 100%); color: white; border: none; padding: 10px 24px; font-size: 14px; font-weight: bold; border-radius: 30px; cursor: pointer; box-shadow: 0 4px 15px rgba(34, 197, 94, 0.4);'>"
                            "🔄 Jugar de Nuevo"
                            "</button></div>"
                        )
                        return
                else:
                    renderizar("⚠️ ¡Puerta bloqueada! Debes recoger todas las monedas primero.")
                    return

            pos_jugador[0] = nueva_f
            pos_jugador[1] = nueva_c

            if tablero_activo[nueva_f, nueva_c] == 2:
                puntuacion += 10
                tablero_activo[nueva_f, nueva_c] = 0

    renderizar()

def tick_tiempo():
    global tiempo_restante, juego_activo, juego_iniciado
    if juego_iniciado and juego_activo:
        if tiempo_restante > 0:
            tiempo_restante -= 1
            renderizar()
        else:
            juego_activo = False
            img_lose = f"<div style='margin-bottom: 15px;'><img src='data:image/jpeg;base64,{img3_base64}' style='width: 100%; height: auto; border-radius: 10px; box-shadow: 0 4px 12px rgba(0,0,0,0.5); display: block;'></div>" if img3_base64 else ""

            pantalla_principal.value = (
                "<div style='font-family: \"Segoe UI\", Roboto, Helvetica, sans-serif; background: linear-gradient(135deg, #090d16 100%, #7f1d1d 0%); color: #f8fafc; padding: 30px; border-radius: 16px; text-align: center; border: 2px solid #ef4444; max-width: 480px; box-shadow: 0 20px 40px rgba(0,0,0,0.6); box-sizing: border-box;'>"
                + img_lose +
                "<h1 style='color: #ef4444; font-size: 26px; margin-bottom: 8px; text-shadow: 0 2px 10px rgba(239,68,68,0.4);'>⏰ ¡HAS PERDIDO! ⏰</h1>"
                "<p style='font-size: 14px; color: #cbd5e1; margin-bottom: 16px;'>El cronómetro llegó a cero y no alcanzaste a terminar el nivel.</p>"
                "<div style='background: rgba(15, 23, 42, 0.85); padding: 14px; border-radius: 10px; margin: 15px 0; border: 1px solid rgba(255,255,255,0.1);'>"
                f"<h3 style='margin: 0 0 5px 0; color: #38bdf8;'>Puntuación acumulada: {puntuacion} pts</h3>"
                "</div>"
                "<button onclick=\"google.colab.kernel.invokeFunction('notebook.iniciar_juego', [], {})\" "
                "style='background: linear-gradient(135deg, #ef4444 0%, #b91c1c 100%); color: white; border: none; padding: 10px 24px; font-size: 14px; font-weight: bold; border-radius: 30px; cursor: pointer; box-shadow: 0 4px 15px rgba(239, 68, 68, 0.4);'>"
                "🔄 Reintentar"
                "</button></div>"
            )

output.register_callback('notebook.mover_teclado', mover_teclado)
output.register_callback('notebook.tick_tiempo', tick_tiempo)
output.register_callback('notebook.iniciar_juego', iniciar_juego)

renderizar()
contenedor_principal.children = [pantalla_principal]
display(contenedor_principal)

display(HTML('''
<script>
if (!window.keyListenerInitialized) {
    window.keyListenerInitialized = true;
    document.addEventListener('keydown', function(e) {
        if(["ArrowUp","ArrowDown","ArrowLeft","ArrowRight","KeyW","KeyS","KeyA","KeyD"].indexOf(e.code) > -1) {
            e.preventDefault();
            let dir = e.code;
            if(dir === 'KeyW') dir = 'ArrowUp';
            if(dir === 'KeyS') dir = 'ArrowDown';
            if(dir === 'KeyA') dir = 'ArrowLeft';
            if(dir === 'KeyD') dir = 'ArrowRight';
            google.colab.kernel.invokeFunction('notebook.mover_teclado', [dir], {});
        }
    });
}

if (window.gameTimer) { clearInterval(window.gameTimer); }
window.gameTimer = setInterval(function() {
    google.colab.kernel.invokeFunction('notebook.tick_tiempo', [], {});
}, 1000);
</script>
'''))